# Tensile Test (TTO): from data entry to RDF

This notebook shows how to describe a uniaxial tensile test and its measured
results, and convert that description into a standardised, machine-readable
RDF graph using the
[Tensile Test Ontology (TTO)](https://w3id.org/pmd/tto/) and the
[Platform MaterialDigital Core Ontology (PMDCo)](https://w3id.org/pmd/co/).

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.


## How this schema relates to the characterization base

This schema **extends** `characterization/step/PMDCo/` via JSON Schema `$ref`
and `allOf`. Think of it as inheritance: the tensile test schema reuses all
the structural fields of the generic characterization step (specimen input,
test conditions, process chain position) and adds its own result fields on top.

```
characterization/step/PMDCo/          ← base schema
  has_specified_input  (specimen IRI)
  preceded_by          (chain)
  has_process_condition (conditions)

characterization/tensile-test/TTO/    ← this schema (extends base)
  type: tto:TensileTest               overrides root class
  measured_properties                 adds result nodes (TTO classes)
```


## What the notebook does

```
example.input.json
  │  specimen IRI, test conditions, measured properties
  │
  ▼
Transform
  │  converts plain JSON to a structured OO-LD document
  │
  ▼
RDF graph
  │  machine-readable, ontology-aligned triples
  │
  ▼
SHACL validation  →  confirms the graph is structurally correct
SPARQL inspect    →  shows the test results captured in the graph
```


## Environment setup

If you are viewing this notebook on GitHub and want to run it locally, clone
the repository first so that all schema files and example inputs are present:

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
```

Then create a virtual environment and start Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

Open this notebook from the `docs/` subfolder inside JupyterLab.

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE       = pathlib.Path().resolve()          # docs/
SCHEMA     = HERE.parent                       # tensile-test/TTO/

schema = Schema(SCHEMA)

print("Schema :", "/".join(SCHEMA.parts[-3:]))


Schema : step/tensile-test/TTO


## Step 1: Describe your tensile test

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `test_name` | yes | A name for this test |
| `specimen_iri` | yes | IRI of the specimen tested (must exist in the knowledge graph) |
| `test_standard` | no | Standard followed, e.g. `"ISO 6892-1"` |
| `strain_rate` | no | Strain or crosshead displacement rate |
| `strain_rate_unit` | no | Unit for strain rate; defaults to `"1/s"` |
| `temperature` | no | Test temperature in °C |
| `results` | no | List of measured properties: `property`, `value`, `unit` |

Supported property names: `YieldStrength`, `TensileStrength`,
`PercentageElongationAfterFracture`, `PercentageReductionOfArea`,
`UpperYieldStrength`, `LowerYieldStrength`, `ProofStrength`,
`PercentagePermanentElongation`.

Supported units: `"MPa"`, `"GPa"` for stress properties; `"%"` for
elongation and reduction of area.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "test_name": "Tensile test 316L bar 1",
  "specimen_iri": "https://example.org/specimens/316L-tensile-bar-1",
  "test_standard": "ISO 6892-1",
  "strain_rate": 0.00025,
  "strain_rate_unit": "1/s",
  "temperature": 23,
  "results": [
    {
      "property": "YieldStrength",
      "value": 310,
      "unit": "MPa"
    },
    {
      "property": "TensileStrength",
      "value": 620,
      "unit": "MPa"
    },
    {
      "property": "PercentageElongationAfterFracture",
      "value": 40,
      "unit": "%"
    },
    {
      "property": "PercentageReductionOfArea",
      "value": 65,
      "unit": "%"
    }
  ]
}


## Step 2: Convert to OO-LD

The transform converts your plain input into a structured OO-LD document.
Because this schema **extends** the characterization base, the converter
produces a single merged document that contains both the inherited fields
(specimen input, test conditions) and the new tensile-test-specific fields
(measured properties typed to TTO classes).

Each result entry becomes a node typed to the corresponding TTO class
(e.g. `tto:YieldStrength`), with a numeric value and a QUDT unit IRI.

In [4]:
oold_doc = schema.transform(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/tensile-test/TTO/#v2.0.0",
  "type": "tto:TensileTest",
  "id": "tensile-test-tensile-test-316l-bar-1",
  "label": "Tensile test 316L bar 1",
  "has_specified_input": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-bar-1-condition-standard",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-bar-1-condition-strain-rate",
      "parameter_label": "Strain Rate",
      "parameter_value": 0.00025,
      "parameter_unit": "1/s"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "tensile-test-tensile-test-316l-bar-1-condition-temperature",
      "parameter_label": "Temperature",
      "parameter_value": 23,
      "parameter_un

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`. The context maps every field to its precise
ontology IRI, for example:

> **Tip:** Pass your own namespace as `base` to `to_graph()` so the generated
> IRIs use your data namespace instead of the schema's internal `@base`.
> The cell below uses `"https://example.org/"` as a placeholder.

| JSON field | Ontology IRI |
|---|---|
| `type` | `rdf:type` |
| `has_specified_input` | `OBI_0000293` |
| `measured_properties` | `OBI_0000299` (has_specified_output) |
| `result_value` | `OBI_0001937` (has_specified_numeric_value) |
| `result_unit` | `IAO_0000039` (has_measurement_unit_label) |

In [5]:
BASE_IRI = "https://example.org/"  # replace with your own namespace in production

flat = schema.to_graph(simplified, base=BASE_IRI)

print(f'Graph contains {len(flat)} triples.\n')
print(flat.serialize(format='turtle'))

Graph contains 34 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tto: <https://w3id.org/pmd/tto/> .
@prefix uqudt: <https://qudt.org/vocab/unit/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://example.org/tensile-test-tensile-test-316l-bar-1> a tto:TensileTest ;
    rdfs:label "Tensile test 316L bar 1" ;
    obo:OBI_0000293 <https://example.org/specimens/316L-tensile-bar-1> ;
    obo:OBI_0000299 <https://example.org/tensile-test-tensile-test-316l-bar-1-result-percentageelongationafterfracture>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-result-percentagereductionofarea>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-result-tensilestrength>,
        <https://example.org/tensile-test-tensile-test-316l-bar-1-result-yield

In [6]:
out_ttl = HERE / "output_tensile_test.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_tensile_test.ttl


## Step 4: Validate against SHACL shapes

Two shape files are loaded:

| Shape file | Validates |
|---|---|
| `characterization/step/PMDCo/specs/shape.ttl` | Assay label, specimen input, process conditions |
| `characterization/tensile-test/TTO/specs/shape.ttl` | Tensile test: specimen required, result values and units |

This mirrors the schema inheritance: the base shape validates the inherited
structure, and the specialised shape validates the tensile-test additions.

In [7]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 5: Inspect the results

The SPARQL query below extracts the mechanical properties captured in the
graph and prints them as a table.

You do not need to understand SPARQL to use this notebook. Just run the cell
and check that the values match what you entered.

In [8]:
TTO  = rdflib.Namespace("https://w3id.org/pmd/tto/")
OBI  = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")

proc_iri = next(flat.subjects(rdflib.RDF.type, TTO["TensileTest"]))
print(f"Test IRI : {proc_iri}")
print(f"Label    : {flat.value(proc_iri, rdflib.RDFS.label)}")
print()

SPARQL = """
PREFIX obi:   <http://purl.obolibrary.org/obo/OBI_>
PREFIX iao:   <http://purl.obolibrary.org/obo/IAO_>
PREFIX tto:   <https://w3id.org/pmd/tto/>
PREFIX uqudt: <https://qudt.org/vocab/unit/>

SELECT ?propertyType ?value ?unit
WHERE {
  ?test  a              tto:TensileTest ;
         obi:0000299    ?result .
  ?result a             ?propertyType ;
          obi:0001937   ?value ;
          iao:0000039   ?unit .
  FILTER(?propertyType != obi:0000299)
}
ORDER BY ?propertyType
"""

rows = list(flat.query(SPARQL))
if rows:
    print(f"{'Property (TTO class)':<45}  {'Value':>8}  Unit")
    print(f"{'':45}  {'':8}  {'':20}")
    for r in rows:
        prop = str(r.propertyType).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        unit = str(r.unit).rsplit("/", 1)[-1]
        print(f"{prop:<45}  {float(r.value):>8.3f}  {unit}")
else:
    print("No measured properties found in the graph.")

Test IRI : https://example.org/tensile-test-tensile-test-316l-bar-1
Label    : Tensile test 316L bar 1



Property (TTO class)                              Value  Unit
                                                                             
PercentageElongationAfterFracture                40.000  PERCENT
PercentageReductionOfArea                        65.000  PERCENT
TensileStrength                                 620.000  MegaPascal
YieldStrength                                   310.000  MegaPascal


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the test name, specimen IRI, conditions, and results |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against two SHACL shape files (base + tensile-test) |
| 5 | The measured properties are extracted from the graph to confirm correctness |

To test a different specimen, edit `docs/example.input.json` and re-run all cells.

---

## What comes next

This notebook used a hand-crafted JSON file as input. In practice, your measurement
data lives in an instrument export file, not a JSON you wrote by hand.

[Notebook 2: from machine file to RDF](2_tensile_test_csv_workflow.ipynb) shows how
to replace the manual input with a real Zwick/Roell export. The schema, the transform,
the SHACL shapes, and the RDF structure are identical — only the data source changes.

---

## Further reading

- [Characterization Step (PMDCo)](../../step/PMDCo/README.md): the base schema this one extends
- [TTO application ontology](https://github.com/materialdigital/application-ontologies/tree/main/tensile_test_ontology_TTO)
- [OO-LD primer](../../../../docs/2_oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../../docs/3_schema-format.md): folder structure and naming conventions
